In [0]:
# ============================================================
# Config
# ============================================================
from pyspark.sql.functions import col, count, when, current_timestamp, lit, round

CATALOG       = "invoices"
SILVER_TABLE  = "invoices.silver.invoices_clean"
DQ_TABLE      = "invoices.silver.dq_report"
BATCH_ID      = "batch_1"

# Read only current batch from Silver — not full table
df = (
    spark.table(SILVER_TABLE)
    .filter(col("batch_id") == BATCH_ID)
)

total_rows = df.count()
print(f"Rows in current batch: {total_rows}")

In [0]:
# ============================================================
# Run all DQ checks in one pass (efficient — single scan)
# ============================================================

dq_metrics = df.select(
    # Null checks
    count(when(col("invoice_number").isNull(),    1)).alias("null_invoice_number"),
    count(when(col("client_name").isNull(),       1)).alias("null_client_name"),
    count(when(col("seller_name").isNull(),       1)).alias("null_seller_name"),
    count(when(col("invoice_date").isNull(),      1)).alias("null_invoice_date"),
    count(when(col("total").isNull(),             1)).alias("null_total"),
    count(when(col("item_description").isNull(),  1)).alias("null_item_description"),
    count(when(col("item_quantity").isNull(),     1)).alias("null_item_quantity"),
    count(when(col("item_total_price").isNull(),  1)).alias("null_item_total_price"),

    # Value checks
    count(when(col("total") < 0,                 1)).alias("negative_total"),
    count(when(col("item_quantity") <= 0,         1)).alias("zero_or_neg_quantity"),
    count(when(col("item_total_price") < 0,       1)).alias("negative_item_price"),

    # Date checks
    count(when((col("invoice_date") > col("invoice_due_date"))
              & col("invoice_due_date").isNotNull(), 1)).alias("due_before_invoice_date"),

    # Empty string checks
    count(when(col("invoice_number") == "",       1)).alias("empty_invoice_number"),
    count(when(col("client_name") == "",          1)).alias("empty_client_name"),
    count(when(col("bank_name") == "",            1)).alias("empty_bank_name"),
    count(when(col("payment_method") == "",       1)).alias("empty_payment_method"),
).collect()[0]

print("DQ metrics collected.")
print(dq_metrics)

In [0]:
# ============================================================
# Build DQ report as structured rows
# ============================================================

from pyspark.sql import Row

dq_rows = [
    Row(check_name="null_invoice_number",    category="Completeness", failed_rows=dq_metrics["null_invoice_number"]),
    Row(check_name="null_client_name",       category="Completeness", failed_rows=dq_metrics["null_client_name"]),
    Row(check_name="null_seller_name",       category="Completeness", failed_rows=dq_metrics["null_seller_name"]),
    Row(check_name="null_invoice_date",      category="Completeness", failed_rows=dq_metrics["null_invoice_date"]),
    Row(check_name="null_total",             category="Completeness", failed_rows=dq_metrics["null_total"]),
    Row(check_name="null_item_description",  category="Completeness", failed_rows=dq_metrics["null_item_description"]),
    Row(check_name="null_item_quantity",     category="Completeness", failed_rows=dq_metrics["null_item_quantity"]),
    Row(check_name="null_item_total_price",  category="Completeness", failed_rows=dq_metrics["null_item_total_price"]),
    Row(check_name="negative_total",         category="Validity",     failed_rows=dq_metrics["negative_total"]),
    Row(check_name="zero_or_neg_quantity",   category="Validity",     failed_rows=dq_metrics["zero_or_neg_quantity"]),
    Row(check_name="negative_item_price",    category="Validity",     failed_rows=dq_metrics["negative_item_price"]),
    Row(check_name="due_before_invoice",     category="Consistency",  failed_rows=dq_metrics["due_before_invoice_date"]),
    Row(check_name="empty_invoice_number",   category="Completeness", failed_rows=dq_metrics["empty_invoice_number"]),
    Row(check_name="empty_client_name",      category="Completeness", failed_rows=dq_metrics["empty_client_name"]),
    Row(check_name="empty_bank_name",        category="Completeness", failed_rows=dq_metrics["empty_bank_name"]),
    Row(check_name="empty_payment_method",   category="Completeness", failed_rows=dq_metrics["empty_payment_method"]),
]

df_dq = (
    spark.createDataFrame(dq_rows)
    .withColumn("total_rows",    lit(total_rows))
    .withColumn("passed_rows",   lit(total_rows) - col("failed_rows"))
    .withColumn("pass_rate_pct", round((col("passed_rows") / col("total_rows")) * 100, 2))
    .withColumn("status",        when(col("failed_rows") == 0, "PASS").otherwise("FAIL"))
    .withColumn("batch_id",      lit(BATCH_ID))
    .withColumn("run_timestamp", current_timestamp())
)

display(df_dq)

In [0]:
# ============================================================
# Persist DQ report
# ============================================================

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

(
    df_dq
    .write
    .format("delta")
    .mode("append")                           # append — keep history of every DQ run
    .saveAsTable(DQ_TABLE)
)

print(f"DQ report written to {DQ_TABLE}")

# Summary — how many checks passed vs failed?
display(
    spark.table(DQ_TABLE)
    .filter(col("batch_id") == BATCH_ID)
    .groupBy("status")
    .count()
)

Check	Failed	Insight
empty_bank_name	5593/5598 (99.9%)	Almost no invoices have bank details — likely cash/card payments
empty_payment_method	5598/5598 (100%)	Payment method field is never filled in the source data